# Inventory Replenishment with PPO

## Goal

이 Notebook은 단일 창고의 보충 의사결정을 PPO(Proximal Policy Optimization)로 학습하는 과정을 단계별로 설명한다.

두 가지 환경을 비교한다.

1. 발주량이 즉시 입고되는 기본 환경
2. 발주 후 3일 뒤 입고되는 Lead-Time 환경

현재 코드는 Gymnasium과 Stable-Baselines3의 작동을 이해하기 위한 교육용 실험이다. 실제 IO Engine에 적용하려면 Forecast, Backorder, MOQ, Lot Multiple, 주문비용, Capacity 제약과 별도 검증 절차가 추가되어야 한다.

## 1. Setup

### 실행 환경 확인

실험에 필요한 패키지를 불러오고 실제 Notebook Kernel의 Python 경로와 라이브러리 버전을 확인한다. PyCharm에서는 Project Interpreter가 `/opt/anaconda3/envs/ai_env/bin/python`인지 확인한다.

모든 실험에서 같은 Seed를 사용해 수요 생성과 PPO 초기화를 재현 가능하게 유지한다.

In [1]:
import sys

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import stable_baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env


SEED = 42
TRAINING_TIMESTEPS = 300_000
TEST_DAYS = 15

print(f"Python: {sys.executable}")
print(f"Gymnasium: {gym.__version__}")
print(f"Stable-Baselines3: {stable_baselines3.__version__}")
print(f"NumPy: {np.__version__}")

Python: /opt/anaconda3/envs/ai_env/bin/python
Gymnasium: 1.3.0
Stable-Baselines3: 2.9.0
NumPy: 2.4.0


## 2. Baseline: 즉시 입고 환경

### 환경 가정

- 상태: 현재 재고를 최대 Capacity로 나눈 값 1개
- 행동: PPO의 `[-1, 1]` 출력을 실제 발주량 `[0, 500]`으로 변환
- 수요: 일평균 100, 표준편차 30인 정규분포에서 생성하고 음수는 0으로 제한
- 입고: 오늘 결정한 발주량이 즉시 재고에 반영
- 비용: 기말 재고의 보유비용 또는 품절 수량의 품절비용
- Episode: 최대 365일

`reset(seed=...)`의 Seed가 실제 수요 생성에도 적용되도록 전역 `np.random` 대신 Gymnasium의 `self.np_random`을 사용한다.

In [3]:
class ImprovedWarehouseEnv(gym.Env):
    # Single-item warehouse environment with immediate replenishment.

    metadata = {"render_modes": []}

    def __init__(self) -> None:
        super().__init__()
        self.max_capacity = 1_000.0
        self.max_order = 500.0
        self.holding_cost = 2.0
        self.stockout_penalty = 10.0
        self.max_days = 365

        self.action_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(1,),
            dtype=np.float32,
        )
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(1,),
            dtype=np.float32,
        )

        self.current_stock = 0.0
        self.current_day = 0

    def _get_obs(self) -> np.ndarray:
        return np.array(
            [self.current_stock / self.max_capacity],
            dtype=np.float32,
        )

    def _decode_action(self, action: np.ndarray) -> float:
        normalized_action = (np.clip(action[0], -1.0, 1.0) + 1.0) / 2.0
        return float(normalized_action * self.max_order)

    def reset(
        self,
        *,
        seed: int | None = None,
        options: dict | None = None,
    ) -> tuple[np.ndarray, dict]:
        super().reset(seed=seed)
        self.current_stock = self.max_capacity / 2.0
        self.current_day = 0
        return self._get_obs(), {}

    def step(
        self,
        action: np.ndarray,
    ) -> tuple[np.ndarray, float, bool, bool, dict]:
        self.current_day += 1
        beginning_stock = self.current_stock
        order_qty = self._decode_action(action)
        demand = max(
            0.0,
            float(self.np_random.normal(loc=100.0, scale=30.0)),
        )
        projected_stock = beginning_stock + order_qty - demand

        if projected_stock >= 0.0:
            self.current_stock = min(projected_stock, self.max_capacity)
            raw_reward = -(self.current_stock * self.holding_cost)
        else:
            self.current_stock = 0.0
            raw_reward = -(abs(projected_stock) * self.stockout_penalty)

        scaled_reward = float(raw_reward / 1_000.0)
        terminated = False
        truncated = self.current_day >= self.max_days
        info = {
            "beginning_stock": float(beginning_stock),
            "received": float(order_qty),
            "demand": float(demand),
            "order": float(order_qty),
            "ending_stock": float(self.current_stock),
            "raw_reward": float(raw_reward),
        }

        return self._get_obs(), scaled_reward, terminated, truncated, info

### 2.1 환경 계약 검사

`check_env`는 Observation·Action Space와 `reset`·`step` 반환 형식이 Gymnasium 계약을 따르는지 검사한다. 이후 고정 Seed로 초기 상태를 확인한다.

In [4]:
baseline_env = ImprovedWarehouseEnv()
check_env(baseline_env)

baseline_initial_obs, _ = baseline_env.reset(seed=SEED)
print("Baseline environment check passed")
print("Initial observation:", baseline_initial_obs)

Baseline environment check passed
Initial observation: [0.5]


### 2.2 PPO 학습

PPO는 연속형 발주 행동을 학습한다. `ent_coef=0.05`는 초기 학습에서 다양한 발주량을 탐색하도록 유도하고, `seed`는 모델 초기화와 Sampling의 재현성을 높인다.

`TRAINING_TIMESTEPS`를 줄이면 빠르게 실행할 수 있지만 학습 결과가 불안정해질 수 있다.

In [5]:
print("--- Baseline PPO 학습 시작 ---")

baseline_model = PPO(
    "MlpPolicy",
    baseline_env,
    ent_coef=0.05,
    learning_rate=0.0003,
    seed=SEED,
    device="cpu",
    verbose=0,
)
baseline_model.learn(total_timesteps=TRAINING_TIMESTEPS)

print("--- Baseline PPO 학습 완료 ---")

--- Baseline PPO 학습 시작 ---
--- Baseline PPO 학습 완료 ---


### 2.3 기본 환경 평가

학습에 사용한 환경을 고정 Seed로 초기화하고 15일간 Deterministic Action을 평가한다. 출력의 재고는 처리 전 `기초재고`와 처리 후 `기말재고`로 구분한다.

기본 환경에서는 발주량이 당일 입고되므로 `입고량`과 `발주량`이 동일하다.

In [6]:
baseline_obs, _ = baseline_env.reset(seed=SEED)

print("--- Baseline PPO 15일 평가 ---")
for day in range(1, TEST_DAYS + 1):
    baseline_action, _ = baseline_model.predict(
        baseline_obs,
        deterministic=True,
    )
    baseline_obs, reward, terminated, truncated, info = baseline_env.step(
        baseline_action
    )

    print(
        f"Day {day:02d} | "
        f"기초재고: {info['beginning_stock']:6.0f} | "
        f"입고량: {info['received']:5.0f} | "
        f"수요: {info['demand']:4.0f} | "
        f"발주량: {info['order']:5.0f} | "
        f"기말재고: {info['ending_stock']:6.0f} | "
        f"비용: {abs(info['raw_reward']):6.0f}"
    )

    if terminated or truncated:
        break

--- Baseline PPO 15일 평가 ---
Day 01 | 기초재고:    500 | 입고량:     0 | 수요:  109 | 발주량:     0 | 기말재고:    391 | 비용:    782
Day 02 | 기초재고:    391 | 입고량:     0 | 수요:   69 | 발주량:     0 | 기말재고:    322 | 비용:    644
Day 03 | 기초재고:    322 | 입고량:     0 | 수요:  123 | 발주량:     0 | 기말재고:    200 | 비용:    399
Day 04 | 기초재고:    200 | 입고량:     0 | 수요:  128 | 발주량:     0 | 기말재고:     71 | 비용:    143
Day 05 | 기초재고:     71 | 입고량:    56 | 수요:   41 | 발주량:    56 | 기말재고:     86 | 비용:    172
Day 06 | 기초재고:     86 | 입고량:    41 | 수요:   61 | 발주량:    41 | 기말재고:     66 | 비용:    133
Day 07 | 기초재고:     66 | 입고량:    62 | 수요:  104 | 발주량:    62 | 기말재고:     24 | 비용:     48
Day 08 | 기초재고:     24 | 입고량:   106 | 수요:   91 | 발주량:   106 | 기말재고:     39 | 비용:     78
Day 09 | 기초재고:     39 | 입고량:    90 | 수요:   99 | 발주량:    90 | 기말재고:     29 | 비용:     59
Day 10 | 기초재고:     29 | 입고량:   100 | 수요:   74 | 발주량:   100 | 기말재고:     55 | 비용:    110
Day 11 | 기초재고:     55 | 입고량:    73 | 수요:  126 | 발주량:    73 | 기말재고:      2 | 비용:      4
Day 12 | 기초재고: 

## 3. Lead-Time 3일 환경

### 환경 확장

발주량이 즉시 입고되지 않고 3일 후 도착하도록 In-transit Pipeline을 추가한다.

- 상태: 현재 재고와 앞으로 3일 동안 도착할 예정 수량
- 행동: 오늘 발주할 수량
- 입고: Pipeline의 첫 번째 수량이 오늘 입고
- Pipeline 이동: 오늘 입고분 제거 후 신규 발주량을 마지막에 추가
- 수불 순서: 기초재고 + 오늘 입고 - 오늘 수요 = 기말재고

PPO는 현재 재고가 낮더라도 다음 날 대량 입고가 예정되어 있으면 추가 발주를 줄이는 정책을 학습할 수 있다.

In [7]:
class LeadTimeWarehouseEnv(gym.Env):
    # Single-item warehouse environment with a fixed replenishment lead time.

    metadata = {"render_modes": []}

    def __init__(self, lead_time: int = 3) -> None:
        super().__init__()
        if lead_time < 1:
            raise ValueError("lead_time must be at least 1")

        self.max_capacity = 1_000.0
        self.max_order = 500.0
        self.holding_cost = 2.0
        self.stockout_penalty = 10.0
        self.lead_time = lead_time
        self.max_days = 365

        self.action_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(1,),
            dtype=np.float32,
        )
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(1 + self.lead_time,),
            dtype=np.float32,
        )

        self.current_stock = 0.0
        self.in_transit_pipeline: list[float] = []
        self.current_day = 0

    def _get_obs(self) -> np.ndarray:
        normalized_stock = self.current_stock / self.max_capacity
        normalized_pipeline = [
            qty / self.max_order for qty in self.in_transit_pipeline
        ]
        return np.array(
            [normalized_stock, *normalized_pipeline],
            dtype=np.float32,
        )

    def _decode_action(self, action: np.ndarray) -> float:
        normalized_action = (np.clip(action[0], -1.0, 1.0) + 1.0) / 2.0
        return float(normalized_action * self.max_order)

    def reset(
        self,
        *,
        seed: int | None = None,
        options: dict | None = None,
    ) -> tuple[np.ndarray, dict]:
        super().reset(seed=seed)
        self.current_stock = self.max_capacity / 2.0
        self.in_transit_pipeline = [0.0] * self.lead_time
        self.current_day = 0
        return self._get_obs(), {}

    def step(
        self,
        action: np.ndarray,
    ) -> tuple[np.ndarray, float, bool, bool, dict]:
        self.current_day += 1
        beginning_stock = self.current_stock
        order_qty = self._decode_action(action)
        received_qty = self.in_transit_pipeline.pop(0)
        demand = max(
            0.0,
            float(self.np_random.normal(loc=100.0, scale=30.0)),
        )
        projected_stock = beginning_stock + received_qty - demand

        if projected_stock >= 0.0:
            self.current_stock = min(projected_stock, self.max_capacity)
            raw_reward = -(self.current_stock * self.holding_cost)
        else:
            self.current_stock = 0.0
            raw_reward = -(abs(projected_stock) * self.stockout_penalty)

        self.in_transit_pipeline.append(order_qty)

        scaled_reward = float(raw_reward / 1_000.0)
        terminated = False
        truncated = self.current_day >= self.max_days
        info = {
            "beginning_stock": float(beginning_stock),
            "received": float(received_qty),
            "demand": float(demand),
            "order": float(order_qty),
            "ending_stock": float(self.current_stock),
            "raw_reward": float(raw_reward),
            "in_transit": tuple(float(qty) for qty in self.in_transit_pipeline),
        }

        return self._get_obs(), scaled_reward, terminated, truncated, info

### 3.1 Lead-Time 환경 계약 검사

Observation은 `[현재 재고, 1일 후 입고, 2일 후 입고, 3일 후 입고]`의 4차원 벡터다. 각 값이 선언된 `[0, 1]` 범위와 Gymnasium 반환 계약을 만족하는지 검사한다.

In [8]:
lead_time_env = LeadTimeWarehouseEnv(lead_time=3)
check_env(lead_time_env)

lead_time_initial_obs, _ = lead_time_env.reset(seed=SEED)
print("Lead-time environment check passed")
print ("Initial observation:", lead_time_initial_obs)

Lead-time environment check passed
Initial observation: [0.5 0.  0.  0. ]


### 3.2 Lead-Time PPO 학습

기본 환경과 같은 Hyperparameter를 사용해 환경 구조 변화의 영향을 비교한다. 모델 객체를 `lead_time_model`로 분리해 기본 환경의 모델과 혼동하지 않도록 한다.

In [9]:
print("--- Lead-Time PPO 학습 시작 ---")

lead_time_model = PPO(
    "MlpPolicy",
    lead_time_env,
    ent_coef=0.05,
    learning_rate=0.0003,
    seed=SEED,
    device="cpu",
    verbose=0,
)
lead_time_model.learn(total_timesteps=TRAINING_TIMESTEPS)

print("--- Lead-Time PPO 학습 완료 ---")

--- Lead-Time PPO 학습 시작 ---
--- Lead-Time PPO 학습 완료 ---


### 3.3 Lead-Time 환경 평가

고정 Seed로 15일간 평가한다. `입고량`은 3일 전에 발주한 수량이며 `발주량`은 오늘 Pipeline에 추가되어 3일 뒤 도착할 수량이다.

기존 마지막 셀의 `KeyError: 'raw_reward'`는 `step()`이 `info`에 `raw_reward`를 넣지 않았지만 출력 코드가 해당 Key를 조회해서 발생했다. 수정된 환경은 원시 비용을 `info['raw_reward']`로 명시적으로 반환한다.

In [10]:
lead_time_obs, _ = lead_time_env.reset(seed=SEED)

print("--- Lead-Time PPO 15일 평가 ---")
for day in range(1, TEST_DAYS + 1):
    lead_time_action, _ = lead_time_model.predict(
        lead_time_obs,
        deterministic=True,
    )
    lead_time_obs, reward, terminated, truncated, info = lead_time_env.step(
        lead_time_action
    )

    print(
        f"Day {day:02d} | "
        f"기초재고: {info['beginning_stock']:6.0f} | "
        f"입고량: {info['received']:5.0f} | "
        f"수요: {info['demand']:4.0f} | "
        f"발주량: {info['order']:5.0f} | "
        f"기말재고: {info['ending_stock']:6.0f} | "
        f"비용: {abs(info['raw_reward']):6.0f}"
    )

    if terminated or truncated:
        break

--- Lead-Time PPO 15일 평가 ---
Day 01 | 기초재고:    500 | 입고량:     0 | 수요:  109 | 발주량:    50 | 기말재고:    391 | 비용:    782
Day 02 | 기초재고:    391 | 입고량:     0 | 수요:   69 | 발주량:    59 | 기말재고:    322 | 비용:    644
Day 03 | 기초재고:    322 | 입고량:     0 | 수요:  123 | 발주량:    59 | 기말재고:    200 | 비용:    399
Day 04 | 기초재고:    200 | 입고량:    50 | 수요:  128 | 발주량:    87 | 기말재고:    121 | 비용:    242
Day 05 | 기초재고:    121 | 입고량:    59 | 수요:   41 | 발주량:    97 | 기말재고:    139 | 비용:    278
Day 06 | 기초재고:    139 | 입고량:    59 | 수요:   61 | 발주량:    72 | 기말재고:    137 | 비용:    273
Day 07 | 기초재고:    137 | 입고량:    87 | 수요:  104 | 발주량:    76 | 기말재고:    120 | 비용:    240
Day 08 | 기초재고:    120 | 입고량:    97 | 수요:   91 | 발주량:    87 | 기말재고:    126 | 비용:    253
Day 09 | 기초재고:    126 | 입고량:    72 | 수요:   99 | 발주량:    83 | 기말재고:     99 | 비용:    197
Day 10 | 기초재고:     99 | 입고량:    76 | 수요:   74 | 발주량:    91 | 기말재고:    100 | 비용:    200
Day 11 | 기초재고:    100 | 입고량:    87 | 수요:  126 | 발주량:    84 | 기말재고:     61 | 비용:    122
Day 12 | 기초재고:

## 4. Checks & Interpretation

### 확인할 내용

- 기본 환경에서 발주량은 당일 재고에 즉시 반영
- Lead-Time 환경에서 초기 3일간 입고량은 0
- 4일차부터 과거 발주량이 순서대로 입고
- 같은 Seed를 사용하면 수요 Sequence 재현 가능
- 모든 Observation이 선언한 `[0, 1]` 범위 유지
- 평가 셀에서 `raw_reward` Key 오류 미발생

### 해석 주의사항

두 실험은 환경 구조를 이해하기 위한 예제이며 운영 성능을 입증하지 않는다. 학습 환경과 평가 환경의 수요가 같은 확률분포를 사용하고 있으며, 별도의 Holdout Scenario나 결정론적 ROP 정책과의 비교가 없다.

## 5. Next Steps

실제 InventoryEngine의 RL 후보 정책으로 발전시키려면 다음 작업이 필요하다.

1. 실제 PSI Simulator를 Gymnasium Adapter가 호출하도록 재고 계산 로직 단일화
2. Forecast Horizon, Backorder, Scheduled Receipt, Lead Time, 수요 변동성을 Observation에 반영
3. MOQ, Lot Multiple, 발주 금지와 Capacity를 강제하는 Action Guardrail 추가
4. 보유비용, 주문비용, 품절비용과 서비스 수준을 포함한 Reward 계약 확정
5. Deterministic ROP 정책과 PPO를 동일 Scenario에서 비교하는 평가 Harness 구축
6. 검증 전까지 `Deterministic Active + PPO Shadow` 방식 유지